# Data-driven and physics-informed neural networks

## Data preparation and five-fold model selection
This notebook contains reproducible data inspection, group-disjoint partitioning, and a predefined random search for the data-driven neural network. The physics-informed model and final test evaluation are deferred. The academic report is maintained separately and is not generated by this notebook.

Only x1–x9 are model inputs. Sample IDs and predictor groups are retained exclusively as metadata. Random seeds, search settings, fold results, and software versions are saved with the outputs. OpenAI Codex assisted with implementation and explanatory text; the author must verify and disclose this contribution.


In [ ]:
%pip install -q numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 xlrd==2.0.1 matplotlib==3.10.8
%pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cpu


## 1. Data source and output directories
Upload the supplied Database.xls in Colab, or set WORKBOOK to a local path. Outputs are separated by analysis stage. Colab runtime files are temporary; download the output archive before ending the session.


In [ ]:
from pathlib import Path

# Change this path when running outside Colab.
WORKBOOK = Path("Database.xls")
if not WORKBOOK.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Set WORKBOOK to your Database.xls path.") from exc
    uploaded = files.upload()
    if "Database.xls" not in uploaded:
        raise FileNotFoundError("Please upload the supplied Database.xls workbook.")
    del uploaded

OUTPUT_ROOT = Path("outputs")
for folder in [OUTPUT_ROOT / "preprocessing", OUTPUT_ROOT / "partitions", OUTPUT_ROOT / "data_driven_search"]:
    folder.mkdir(parents=True, exist_ok=True)

## 2. Inspect and prepare the data
Read the second header row, validate numeric values and IDs, retain every observation, and identify exact repeated predictor combinations. Standardization is defined here but remains unfitted. The initial target representation is log10(y); its inverse yields positive predictions. This is separate from the RMSLE metric definition. Original numerical units must be used in the physics equation.

In [ ]:
"""Inspect the supplied workbook without fitting transforms on held-out data."""
from pathlib import Path
import argparse
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

FEATURES = [f"x{i}" for i in range(1, 10)]


def load_workbook(path):
    # Row 1 contains merged section headings; row 2 contains feature names.
    df = pd.read_excel(path, sheet_name="sorted all", header=1)
    if df.shape[1] != 11 or list(df.columns[1:]) != FEATURES + ["y"]:
        raise ValueError("Unexpected workbook columns; inspect the source header.")
    df.columns = ["ID"] + FEATURES + ["y"]
    df = df.apply(pd.to_numeric, errors="raise")
    if not np.isfinite(df.to_numpy()).all():
        raise ValueError("Missing or nonfinite values require explicit investigation.")
    if df.ID.duplicated().any() or (df.ID % 1 != 0).any():
        raise ValueError("IDs must be unique integers.")
    if (df[["x1", "x2", "y"]] <= 0).any().any():
        raise ValueError("The physical relation and log target require positive x1, x2, y.")
    df["ID"] = df.ID.astype("int64")
    # Exact predictor matches share an ID regardless of their observed target.
    df["predictor_group"] = df.groupby(FEATURES, sort=True, dropna=False).ngroup()
    return df


class FoldPreprocessor:
    """Fit only on a fold's training rows; preserve original units separately.

    Raw features are standardized, including features that can be zero.
    The target is represented as log10(y), with no learned target statistics.
    """

    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, training_rows):
        self.scaler.fit(training_rows[FEATURES])
        return self

    def transform_inputs(self, rows):
        return self.scaler.transform(rows[FEATURES])

    @staticmethod
    def transform_target(rows):
        values = rows["y"].to_numpy(dtype=float)
        if not np.isfinite(values).all() or (values <= 0).any():
            raise ValueError("Log target requires finite positive values.")
        return np.log10(values)

    @staticmethod
    def inverse_target(log_predictions):
        with np.errstate(over="raise", under="raise", invalid="raise"):
            predictions = np.power(10.0, np.asarray(log_predictions, dtype=float))
        if not np.isfinite(predictions).all() or (predictions <= 0).any():
            raise ValueError("Predictions must be finite and strictly positive.")
        return predictions


def prepare_preprocessing(path, output):
    df = load_workbook(path)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    groups = df.groupby("predictor_group")
    audit = {
        # Total number of observations.
        "samples": len(df),
        # Names of the nine model input columns.
        "predictors": FEATURES,
        # Total number of missing cells across all columns.
        "missing_cells": int(df.isna().sum().sum()),
        # Total number of NaN or positive/negative infinity cells.
        "nonfinite_cells": int((~np.isfinite(df.to_numpy())).sum()),
        # Number of distinct combinations of all nine input features.
        "unique_predictor_groups": int(groups.ngroups),
        # All rows sharing their feature combination with another row, including the first.
        "rows_in_repeated_predictor_groups": int(df.duplicated(FEATURES, keep=False).sum()),
        # Rows repeating a feature combination after its first occurrence.
        "extra_rows_with_repeated_predictors": int(df.duplicated(FEATURES).sum()),
        # Feature groups containing more than one distinct target value.
        "groups_with_different_targets": int((groups.y.nunique() > 1).sum()),
        # Extra rows with identical features AND target, ignoring sample ID.
        "repeated_records_excluding_id": int(df.duplicated(FEATURES + ["y"]).sum()),
        # No observations are removed by this preprocessing step.
        "removed_rows": 0,
        # Scaling is deferred until training partitions have been constructed.
        "scaling_fitted": False,
    }
    df.to_csv(output / "validated_data.csv", index=False, float_format="%.17g")
    df.describe().to_csv(output / "descriptive_statistics.csv")
    repeated = df[df.duplicated(FEATURES, keep=False)].sort_values(["predictor_group", "ID"])
    repeated.to_csv(output / "repeated_predictors.csv", index=False, float_format="%.17g")
    (output / "data_audit.json").write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2))




In [ ]:
prepare_preprocessing(WORKBOOK, OUTPUT_ROOT / "preprocessing")
display(load_workbook(WORKBOOK).head())

## 3. Shared grouped partitions
Use seed 42. Hold out 20% of distinct predictor groups, then create five shuffled group folds within training. Keep targets out of partition construction. A validation_fold of k means the row is held out in CV round k; train that round on the other four folds. Test rows have no fold. Reuse the saved assignments for both models. Fit a fresh scaler on fitting rows only in each round; do not fit on validation or test rows.

In [ ]:
"""Save shared, group-disjoint holdout and five-fold assignments (seed 42)."""
import argparse
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit



SEED = 42


def create_assignments(df):
    # The fraction applies to predictor groups, so row counts may depart from 80/20.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_indices, test_indices = next(splitter.split(df, groups=df.predictor_group))
    assignments = df[["ID", "predictor_group"]].copy()
    assignments["partition"] = "train"
    assignments.loc[test_indices, "partition"] = "test"
    # Fold numbers indicate when a training row is used for validation.
    # Test rows have no validation fold and must never enter cross-validation.
    assignments["validation_fold"] = pd.Series(pd.NA, index=df.index, dtype="Int64")
    training = df.iloc[train_indices]
    folds = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (_, validation_indices) in enumerate(
        folds.split(training, groups=training.predictor_group), start=1
    ):
        assignments.loc[training.index[validation_indices], "validation_fold"] = fold
    validate_assignments(df, assignments)
    return assignments


def validate_assignments(df, assignments):
    if len(assignments) != len(df) or not assignments.ID.equals(df.ID):
        raise ValueError("Assignments must preserve every sample ID exactly once.")
    if assignments.groupby("predictor_group").partition.nunique().max() != 1:
        raise ValueError("A predictor group crosses the train/test boundary.")
    training = assignments[assignments.partition == "train"]
    testing = assignments[assignments.partition == "test"]
    if training.validation_fold.isna().any() or testing.validation_fold.notna().any():
        raise ValueError("Only training samples must have validation fold assignments.")
    if set(training.validation_fold) != set(range(1, 6)):
        raise ValueError("Exactly five nonempty validation folds are required.")
    if training.groupby("predictor_group").validation_fold.nunique().max() != 1:
        raise ValueError("A predictor group crosses validation folds.")


def prepare_partitions(workbook, output):
    df = load_workbook(workbook)
    assignments = create_assignments(df)
    # Recreate assignments to verify deterministic behavior without using targets.
    pd.testing.assert_frame_equal(assignments, create_assignments(df))
    training = assignments[assignments.partition == "train"]
    testing = assignments[assignments.partition == "test"]
    summary = {
        "seed": SEED,
        "holdout_method": "GroupShuffleSplit(test_size=0.2)",
        "fold_method": "GroupKFold(n_splits=5, shuffle=True)",
        "group_definition": "Exact equality of x1 through x9; target excluded",
        "total_samples": len(df),
        "training_samples": len(training),
        "testing_samples": len(testing),
        "training_fraction": len(training) / len(df),
        "testing_fraction": len(testing) / len(df),
        "training_groups": training.predictor_group.nunique(),
        "testing_groups": testing.predictor_group.nunique(),
        "folds": [
            {"fold": fold,
             "training_samples": int((training.validation_fold != fold).sum()),
             "validation_samples": int((training.validation_fold == fold).sum())}
            for fold in range(1, 6)
        ],
        "checks": {
            "all_samples_assigned_once": True,
            "no_train_test_group_overlap": True,
            "no_cross_fold_group_overlap": True,
            "reproducible_assignments": True,
        },
        "scaling_fitted": False,
        "models_trained": False,
    }
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    assignments.to_csv(output / "partition_assignments.csv", index=False)
    (output / "partition_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps(summary, indent=2))




In [ ]:
assignment_path = OUTPUT_ROOT / "partitions/partition_assignments.csv"
if not assignment_path.exists():
    prepare_partitions(WORKBOOK, OUTPUT_ROOT / "partitions")
else:
    print("Reusing saved partition assignments; their integrity is checked before training.")
display(pd.read_csv(assignment_path).head())


## 4. Experimental design and search configuration

Each candidate is evaluated on the same five saved training folds. Preprocessing is fitted on the fitting portion of each fold only. The held-out test partition is excluded from training, early stopping, and selection.

| Hyperparameter | Candidate values | Rationale |
|---|---|---|
| Hidden layers | 1, 2, 3 | Assess whether additional depth helps model nonlinear relationships. |
| Uniform hidden width | 16, 32, 64 | Compare modest capacities given nine inputs and approximately 1,194 fitting samples per fold. |
| Activation | ReLU, tanh | Compare piecewise-linear and smooth mappings. |
| Learning rate | 0.0003, 0.001, 0.003 | Explore a tenfold range around a conventional Adam starting rate. |
| Batch size | 32, 64, 128 | Compare stochastic update frequency and batch averaging. |
| L2 coefficient | 0, 0.00001, 0.0001, 0.001 | Compare unregularized fitting with increasingly strong weight penalties. |
| Selected input logarithms | enabled, disabled | Assess a compressed representation while retaining zero-valued features unchanged. |

The logged option applies log10 to x1, x2, x4, x5, and x6, which are positive in the supplied workbook. Compression is especially motivated by the wide ranges of x2 and x4; the remaining logs test a representation choice. Standardization follows either representation. Original values remain available for subsequent physics-informed modeling.

Twenty distinct combinations are sampled uniformly without replacement from 1,296 possible combinations using NumPy's seed-42 generator: **100 fold fits**. This is a budget-limited random search, not an exhaustive optimization or a controlled estimate of individual hyperparameter effects. The winner is the best configuration evaluated.

Fixed settings are Adam (β₁=0.9, β₂=0.999, ε=10⁻⁷), a log10(y) target, CPU execution with one PyTorch thread, and deterministic algorithms. Hidden weights use He initialization for ReLU or Xavier initialization with tanh gain; the output layer uses Xavier initialization and all biases start at zero. Fold training seeds are 42 + fold number and are shared across candidates.

The optimization objective is
\[
L=\operatorname{mean}\{(f_\theta(X)-\log_{10}y)^2\}+\alpha\sum_{\text{weight matrices}}\lVert W\rVert_F^2.
\]
Biases are excluded from L2. No additional weight decay or learning-rate schedule is used. This training objective differs from the selection metric.

Validation RMSLE is defined explicitly as
\[
\sqrt{\operatorname{mean}\{[\log_{10}(1+\hat y)-\log_{10}(1+y)]^2\}},\quad \hat y=10^{f_\theta(X)}.
\]
The +1 convention is stated because a log-target RMSE is a different quantity. Invalid numerical predictions raise errors; predictions are never silently clipped.

Training stops after 50 epochs without a cumulative validation improvement of more than 10⁻⁵, or at 1,000 epochs. The absolute minimum validation-RMSLE checkpoint is retained. Candidates are ranked by the mean of five fold scores; sample standard deviation (ddof=1) is also reported. Exact ties use candidate ID. For later full-training refitting, the predetermined duration rule is the median of the selected candidate's five best epochs. Final refitting and test evaluation are not performed in this stage.


In [ ]:
SEARCH_CONFIG = {'seed': 42, 'n_configs': 20, 'n_folds': 5, 'search_strategy': 'random_without_replacement', 'selection_metric': 'mean_validation_rmsle_base10_log1p', 'target_transform': 'log10', 'log_features': ['x1', 'x2', 'x4', 'x5', 'x6'], 'optimizer': {'name': 'adam', 'epsilon': 1e-07, 'beta_1': 0.9, 'beta_2': 0.999}, 'training': {'max_epochs': 1000, 'patience': 50, 'min_delta': 1e-05}, 'search_space': {'n_layers': [1, 2, 3], 'width': [16, 32, 64], 'activation': ['relu', 'tanh'], 'learning_rate': [0.0003, 0.001, 0.003], 'batch_size': [32, 64, 128], 'l2': [0.0, 1e-05, 0.0001, 0.001], 'log_inputs': [True, False]}}

# Save the exact experiment settings before executing any training runs.
config_directory = Path("configs")
config_directory.mkdir(exist_ok=True)
(config_directory / "data_driven_search.json").write_text(json.dumps(SEARCH_CONFIG, indent=2) + "\n")
print(json.dumps(SEARCH_CONFIG, indent=2))

## 5. Cross-validation implementation
The implementation records every sampled candidate, fold checkpoint, fitted scaler statistics, per-epoch history, out-of-fold predictions, elapsed time, and software versions. Completed fold fits can be resumed only when the experiment manifest matches. If changing code, data, or settings, use a new output directory. Checkpoints contain fold models and are not final models refitted on all training observations.

In [ ]:
"""Reproducible five-fold random search; the test partition is never evaluated."""
import argparse
import copy
import hashlib
import itertools
import json
import platform
import random
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.preprocessing import StandardScaler





def rmsle_base10(y, log_prediction):
    # Invert log10(y) outputs without clipping; calculate the metric in float64.
    with np.errstate(over="raise", under="raise", invalid="raise"):
        prediction = np.power(10.0, np.asarray(log_prediction, dtype=np.float64))
    if not np.isfinite(prediction).all() or (prediction <= 0).any():
        raise ValueError("Invalid predictions: no silent clipping is permitted.")
    return float(np.sqrt(np.mean((np.log10(1.0 + prediction) - np.log10(1.0 + y)) ** 2)))


def input_values(rows, candidate, config):
    values = rows[FEATURES].copy()
    if candidate["log_inputs"]:
        names = config["log_features"]
        if (values[names] <= 0).any().any():
            raise ValueError("Selected logarithmic inputs must be positive.")
        values[names] = np.log10(values[names])
    return values


def build_network(candidate):
    layers = []
    incoming = len(FEATURES)
    for _ in range(candidate["n_layers"]):
        layer = torch.nn.Linear(incoming, candidate["width"])
        if candidate["activation"] == "relu":
            torch.nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
        else:
            torch.nn.init.xavier_uniform_(layer.weight, gain=torch.nn.init.calculate_gain("tanh"))
        torch.nn.init.zeros_(layer.bias)
        layers.extend([layer, torch.nn.ReLU() if candidate["activation"] == "relu" else torch.nn.Tanh()])
        incoming = candidate["width"]
    output = torch.nn.Linear(incoming, 1)
    torch.nn.init.xavier_uniform_(output.weight)
    torch.nn.init.zeros_(output.bias)
    return torch.nn.Sequential(*layers, output)


def fit_fold(fitting, validation, candidate, config, fold):
    # The same fold seed is used for every candidate; all randomness is explicit.
    seed = config["seed"] + fold
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    scaler = StandardScaler().fit(input_values(fitting, candidate, config))
    x_fit = torch.tensor(scaler.transform(input_values(fitting, candidate, config)), dtype=torch.float32)
    x_val = torch.tensor(scaler.transform(input_values(validation, candidate, config)), dtype=torch.float32)
    y_fit_raw = fitting.y.to_numpy(dtype=float)
    y_val_raw = validation.y.to_numpy(dtype=float)
    y_fit = torch.tensor(np.log10(y_fit_raw), dtype=torch.float32)
    y_val = torch.tensor(np.log10(y_val_raw), dtype=torch.float32)
    model = build_network(candidate)
    optimizer_config = config["optimizer"]
    optimizer = torch.optim.Adam(model.parameters(), lr=candidate["learning_rate"],
        eps=optimizer_config["epsilon"], betas=(optimizer_config["beta_1"], optimizer_config["beta_2"]))
    controls = config["training"]
    best_score = float("inf")
    patience_anchor = float("inf")
    stale = 0
    history = []
    start = time.perf_counter()
    for epoch in range(1, controls["max_epochs"] + 1):
        model.train()
        permutation = torch.randperm(len(x_fit))
        for indices in permutation.split(candidate["batch_size"]):
            optimizer.zero_grad(set_to_none=True)
            prediction = model(x_fit[indices]).squeeze(-1)
            loss = torch.mean((prediction - y_fit[indices]) ** 2)
            # Explicit L2 on weights only, not biases; no separate weight decay.
            penalty = sum(p.square().sum() for name, p in model.named_parameters() if name.endswith("weight"))
            objective = loss + candidate["l2"] * penalty
            if not torch.isfinite(objective):
                raise ValueError("Nonfinite training objective.")
            objective.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            fit_prediction = model(x_fit).squeeze(-1).numpy()
            val_prediction = model(x_val).squeeze(-1).numpy()
        training_score = rmsle_base10(y_fit_raw, fit_prediction)
        validation_score = rmsle_base10(y_val_raw, val_prediction)
        history.append({"epoch": epoch, "training_rmsle": training_score,
            "validation_rmsle": validation_score,
            "training_log10_mse": float(np.mean((fit_prediction - y_fit.numpy()) ** 2)),
            "validation_log10_mse": float(np.mean((val_prediction - y_val.numpy()) ** 2))})
        # Checkpoint the absolute minimum; min_delta controls patience only.
        if validation_score < best_score:
            best_score = validation_score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_training_score = training_score
        if validation_score < patience_anchor - controls["min_delta"]:
            patience_anchor = validation_score
            stale = 0
        else:
            stale += 1
        if stale >= controls["patience"]:
            break
    model.load_state_dict(best_state)
    with torch.no_grad():
        restored = model(x_val).squeeze(-1).numpy()
    if not np.isclose(rmsle_base10(y_val_raw, restored), best_score, rtol=1e-6):
        raise ValueError("Restored checkpoint does not match its recorded score.")
    checkpoint = {"state_dict": best_state, "candidate": candidate,
        "scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist(),
        "log_features": config["log_features"] if candidate["log_inputs"] else [],
        "target_transform": config["target_transform"], "fold": fold, "seed": seed}
    result = {"fold": fold, "validation_rmsle": best_score,
        "training_rmsle_at_best_epoch": best_training_score,
        "best_epoch": best_epoch, "epochs_run": epoch,
        "hit_epoch_limit": epoch == controls["max_epochs"],
        "seconds": time.perf_counter() - start}
    predictions = pd.DataFrame({"ID": validation.ID.to_numpy(), "fold": fold,
        "observed_y": y_val_raw, "predicted_y": np.power(10.0, restored.astype(float))})
    return result, pd.DataFrame(history), checkpoint, predictions


def save_results_table(results, output):
    """Generate one record per candidate from its five measured fold scores."""
    if results[["config_id", "fold"]].duplicated().any():
        raise ValueError("Duplicate candidate/fold results.")
    if not results.groupby("config_id").fold.apply(lambda x: set(x) == set(range(1, 6))).all():
        raise ValueError("Each candidate must have exactly folds 1 through 5.")
    parameters = ["n_layers", "width", "activation", "learning_rate", "batch_size", "l2", "log_inputs"]
    if (results.groupby("config_id")[parameters].nunique() != 1).any().any():
        raise ValueError("Hyperparameters changed across folds of a candidate.")
    settings = results.groupby("config_id")[parameters].first()
    scores = results.pivot(index="config_id", columns="fold", values="validation_rmsle")
    scores.columns = [f"fold_{fold}_rmsle" for fold in scores.columns]
    summary = results.groupby("config_id").agg(
        mean_validation_rmsle=("validation_rmsle", "mean"),
        std_validation_rmsle=("validation_rmsle", "std"),
        total_training_seconds=("seconds", "sum"))
    table = settings.join(scores).join(summary).reset_index()
    table = table.sort_values(["mean_validation_rmsle", "config_id"]).reset_index(drop=True)
    table.insert(0, "rank", np.arange(1, len(table) + 1))
    table["selected"] = table["rank"] == 1
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    table.to_csv(output / "configuration_results.csv", index=False)
    # A Markdown version makes the same generated results readable on GitHub.
    headers = list(table.columns)
    lines = ["# Data-driven five-fold cross-validation results", "",
        "Twenty candidates; five folds each. Lower RMSLE is better. Scores use base-10 log(1+y).",
        "Standard deviation is across folds (ddof=1). These are validation selection scores; the test set was not evaluated.", "",
        "| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for values in table.itertuples(index=False, name=None):
        formatted = [f"{value:.6g}" if isinstance(value, (float, np.floating)) else str(value) for value in values]
        lines.append("| " + " | ".join(formatted) + " |")
    (output / "configuration_results.md").write_text("\n".join(lines) + "\n")
    return table


def plot_selected_folds(output, selection):
    """Show all five selected-candidate runs without averaging unaligned epochs."""
    output = Path(output)
    candidate_id = selection["candidate"]["config_id"]
    figure, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True)
    for fold, axis in enumerate(axes.flat[:5], start=1):
        curve = pd.read_csv(output / "histories" / f"config_{candidate_id:02d}_fold_{fold}.csv")
        best_row = curve.loc[curve.validation_rmsle.idxmin()]
        axis.plot(curve.epoch, curve.training_rmsle, label="Training", linewidth=1.2)
        axis.plot(curve.epoch, curve.validation_rmsle, label="Validation", linewidth=1.2)
        axis.axvline(best_row.epoch, color="gray", linestyle="--", label="Selected epoch")
        axis.set_title(f"Fold {fold}: best epoch {int(best_row.epoch)}")
        axis.set_xlabel("Epoch")
        axis.set_ylabel("RMSLE (base 10, log1p)")
        axis.tick_params(labelbottom=True, labelleft=True)
    axes.flat[5].axis("off")
    handles, labels = axes.flat[0].get_legend_handles_labels()
    axes.flat[5].legend(handles, labels, loc="center", frameon=False)
    axes.flat[5].text(0.5, 0.24,
        f"Configuration {candidate_id}\nFive-fold validation RMSLE\n"
        f"{selection['mean_validation_rmsle']:.5f} ± {selection['std_validation_rmsle']:.5f}",
        ha="center", va="center", transform=axes.flat[5].transAxes)
    figure.suptitle("Selected data-driven network: all five cross-validation folds")
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    figure.savefig(output / "selected_all_folds_learning_curves.png", dpi=180)
    figure.savefig(output / "selected_all_folds_learning_curves.pdf")
    plt.close(figure)


def run_search(workbook, assignments_path, config, output):
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    output = Path(output)
    for directory in [output, output / "histories", output / "checkpoints", output / "oof_predictions"]:
        directory.mkdir(parents=True, exist_ok=True)
    df = load_workbook(workbook)
    assignments = pd.read_csv(assignments_path, dtype={"validation_fold": "Int64"})
    # Reorder by ID and verify the saved groups still match the supplied workbook.
    if assignments.ID.duplicated().any() or set(assignments.ID) != set(df.ID):
        raise ValueError("Saved assignments do not match workbook IDs.")
    assignments = assignments.set_index("ID").loc[df.ID].reset_index()
    if not np.array_equal(assignments.predictor_group, df.predictor_group):
        raise ValueError("Saved predictor groups do not match the workbook.")
    validate_assignments(df, assignments)
    training = df.loc[assignments.partition == "train"].copy()
    training["validation_fold"] = assignments.loc[assignments.partition == "train", "validation_fold"]
    keys = list(config["search_space"])
    grid = list(itertools.product(*(config["search_space"][key] for key in keys)))
    sampled = np.random.default_rng(config["seed"]).choice(len(grid), config["n_configs"], replace=False)
    candidates = [{"config_id": i + 1, **dict(zip(keys, grid[int(index)]))} for i, index in enumerate(sampled)]
    metadata = {"configuration": config, "candidates": candidates,
        "workbook_sha256": hashlib.sha256(Path(workbook).read_bytes()).hexdigest(),
        "assignments_sha256": hashlib.sha256(Path(assignments_path).read_bytes()).hexdigest(),
        "implementation_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if "__file__" in globals() else "embedded_notebook",
        "versions": {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
        "device": "cpu", "threads": 1,
        "test_evaluated": False}
    metadata_path = output / "search_manifest.json"
    if metadata_path.exists() and json.loads(metadata_path.read_text()) != metadata:
        raise ValueError("Existing results belong to a different run. Use a new output directory.")
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
    records = []
    started = time.perf_counter()
    for candidate in candidates:
        for fold in range(1, config["n_folds"] + 1):
            stem = f"config_{candidate['config_id']:02d}_fold_{fold}"
            record_path = output / "histories" / f"{stem}.json"
            if record_path.exists():
                record = json.loads(record_path.read_text())
                required = [output / "histories" / f"{stem}.csv", output / "checkpoints" / f"{stem}.pt",
                    output / "oof_predictions" / f"{stem}.csv"]
                if not all(p.exists() for p in required):
                    raise ValueError("Incomplete cached fold outputs; use a new output directory.")
            else:
                fitting = training.loc[training.validation_fold != fold]
                validation = training.loc[training.validation_fold == fold]
                if set(fitting.predictor_group) & set(validation.predictor_group):
                    raise ValueError("Predictor group leakage.")
                result, history, checkpoint, predictions = fit_fold(fitting, validation, candidate, config, fold)
                record = {**candidate, **result}
                history.to_csv(output / "histories" / f"{stem}.csv", index=False)
                torch.save(checkpoint, output / "checkpoints" / f"{stem}.pt")
                predictions.to_csv(output / "oof_predictions" / f"{stem}.csv", index=False)
                record_path.write_text(json.dumps(record, indent=2) + "\n")
            records.append(record)
            pd.DataFrame(records).to_csv(output / "fold_results.csv", index=False)
            print(f"Config {candidate['config_id']:02d}/{len(candidates)}, fold {fold}: "
                  f"RMSLE={record['validation_rmsle']:.6f}, best epoch={record['best_epoch']}, "
                  f"run epochs={record['epochs_run']}", flush=True)
    results = pd.DataFrame(records)
    ranking = results.groupby("config_id").agg(
        mean_validation_rmsle=("validation_rmsle", "mean"),
        std_validation_rmsle=("validation_rmsle", "std"),
        mean_best_epoch=("best_epoch", "mean"),
        folds=("fold", "count"), total_seconds=("seconds", "sum"),
        folds_hitting_epoch_limit=("hit_epoch_limit", "sum")).reset_index()
    if len(results) != config["n_configs"] * config["n_folds"] or not (ranking.folds == 5).all():
        raise ValueError("Search incomplete.")
    ranking = ranking.sort_values(["mean_validation_rmsle", "config_id"])
    ranking.to_csv(output / "search_ranking.csv", index=False)
    save_results_table(results, output)
    winning_id = int(ranking.iloc[0].config_id)
    winning = next(c for c in candidates if c["config_id"] == winning_id)
    selected_folds = results[results.config_id == winning_id]
    selection = {"candidate": winning,
        "mean_validation_rmsle": float(ranking.iloc[0].mean_validation_rmsle),
        "std_validation_rmsle": float(ranking.iloc[0].std_validation_rmsle),
        "std_definition": "sample standard deviation across five folds (ddof=1)",
        "fold_best_epochs": selected_folds.best_epoch.tolist(),
        "proposed_final_refit_epochs": int(np.median(selected_folds.best_epoch)),
        "refit_rule": "Median of selected configuration's five best epochs; no test data used",
        "training_seconds_total": float(results.seconds.sum()),
        "current_invocation_seconds": time.perf_counter() - started,
        "completed_fits": len(results), "final_refit_performed": False,
        "test_evaluated": False,
        "selected_folds_hitting_epoch_limit": int(selected_folds.hit_epoch_limit.sum())}
    (output / "selected_configuration.json").write_text(json.dumps(selection, indent=2) + "\n")
    oof = pd.concat([pd.read_csv(output / "oof_predictions" / f"config_{winning_id:02d}_fold_{fold}.csv")
        for fold in range(1, 6)], ignore_index=True)
    if len(oof) != len(training) or set(oof.ID) != set(training.ID) or oof.ID.duplicated().any():
        raise ValueError("Selected out-of-fold predictions do not cover training exactly once.")
    oof.to_csv(output / "selected_oof_predictions.csv", index=False)
    # Representative fold fixed in advance: fold 1, rather than the best-looking fold.
    curve = pd.read_csv(output / "histories" / f"config_{winning_id:02d}_fold_1.csv")
    figure, axis = plt.subplots(figsize=(6, 3.6))
    axis.plot(curve.epoch, curve.training_rmsle, label="Training")
    axis.plot(curve.epoch, curve.validation_rmsle, label="Validation")
    best_epoch = int(selected_folds.loc[selected_folds.fold == 1, "best_epoch"].iloc[0])
    axis.axvline(best_epoch, color="gray", linestyle="--", label="Selected epoch")
    axis.set(xlabel="Epoch", ylabel="RMSLE (base 10, log1p)", title="Data-driven network: selected configuration, fold 1")
    axis.legend()
    figure.tight_layout()
    figure.savefig(output / "selected_fold_1_learning_curve.png", dpi=180)
    plt.close(figure)
    plot_selected_folds(output, selection)
    print(json.dumps(selection, indent=2), flush=True)
    return ranking, selection




## 6. Execute the predefined search
This cell performs 100 fold fits and can take several minutes or longer on CPU. Only training-partition observations are passed into fold fitting. Test data are not scored. Results are written incrementally so progress is retained if execution is interrupted.

In [ ]:
SEARCH_OUTPUT = OUTPUT_ROOT / "data_driven_search"
ranking, selected_configuration = run_search(
    WORKBOOK, assignment_path, SEARCH_CONFIG, SEARCH_OUTPUT
)
display(ranking)
print(json.dumps(selected_configuration, indent=2))

### Complete configuration results
The search code generates both `configuration_results.csv` and `configuration_results.md` from the measured fold scores. Each record contains the hyperparameters, all five validation scores, mean and sample standard deviation, training time, and selection rank. The CSV retains full numerical precision. This table is an experiment record, separate from the academic report.


In [ ]:
configuration_results = pd.read_csv(SEARCH_OUTPUT / "configuration_results.csv")
display(configuration_results)


rank,config_id,n_layers,width,activation,learning_rate,batch_size,l2,log_inputs,fold_1_rmsle,fold_2_rmsle,fold_3_rmsle,fold_4_rmsle,fold_5_rmsle,mean_validation_rmsle,std_validation_rmsle,total_training_seconds,selected
1,2,3,16,relu,0.0030,64,0.00100,True,0.304186,0.307538,0.285252,0.320640,0.280810,0.299685,0.016474,8.049488,True
2,1,3,16,tanh,0.0030,32,0.00010,False,0.300810,0.309528,0.290925,0.324657,0.287301,0.302644,0.015067,13.632061,False
3,4,3,64,tanh,0.0010,64,0.00000,True,0.301724,0.326249,0.285835,0.317819,0.286143,0.303554,0.018296,7.860742,False
4,17,3,32,relu,0.0003,32,0.00100,False,0.300870,0.314884,0.288170,0.326744,0.291243,0.304382,0.016264,28.951906,False
5,11,2,16,tanh,0.0030,32,0.00001,True,0.304554,0.315097,0.295594,0.326862,0.281705,0.304763,0.017399,11.476791,False
6,5,3,32,tanh,0.0003,32,0.00100,False,0.307653,0.318259,0.279795,0.329468,0.289551,0.304945,0.020344,37.400574,False
7,18,2,16,tanh,0.0030,64,0.00000,False,0.307366,0.311826,0.283359,0.336446,0.289372,0.305674,0.020924,7.751380,False
8,19,3,16,relu,0.0010,32,0.00100,False,0.309880,0.325177,0.284627,0.324525,0.285064,0.305855,0.020131,21.617124,False
9,9,2,64,tanh,0.0010,128,0.00010,False,0.313525,0.305133,0.280873,0.336203,0.295100,0.306166,0.020734,7.631331,False
10,20,2,32,tanh,0.0010,32,0.00010,False,0.309644,0.320747,0.282092,0.330385,0.293657,0.307305,0.019627,13.856703,False


## 7. Selected configuration and training diagnostics
The figure shows training and validation RMSLE for all five folds of the selected configuration, with shared axes and each selected epoch marked. Runs end at their actual stopping epochs; curves are not padded or averaged. Configuration selection uses the mean of all five minimum validation scores, not any single fold. No test errors are plotted.

A single optimization seed per fold does not quantify sensitivity to initialization. Fold standard deviation measures variation across partitions, not a confidence interval. Validation selection scores are not an independent generalization estimate. Final testing is reserved for a later stage.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(SEARCH_OUTPUT / "selected_all_folds_learning_curves.png")))
if selected_configuration["selected_folds_hitting_epoch_limit"]:
    print("Investigate epoch-limit convergence before finalizing the training duration.")
else:
    print("All five selected-configuration folds stopped before the epoch ceiling.")

## Recorded cloud execution results

The predefined 20-candidate search completed all 100 fold fits using the implementation above. These are cross-validation selection results, not test results. Approximate cumulative fold-training time was 4.89 minutes on one CPU thread.

| Selected setting | Value |
|---|---|
| Hidden layers | 16 → 16 → 16 |
| Activation | ReLU |
| Learning rate | 0.003 |
| Batch size | 64 |
| L2 coefficient | 0.001 |
| Selected input logs | Enabled |
| Mean validation RMSLE | 0.299685 |
| Fold standard deviation | 0.016474 |

| Fold | Best epoch | Validation RMSLE |
|---|---|---|
| 1 | 31 | 0.304186 |
| 2 | 123 | 0.307538 |
| 3 | 89 | 0.285252 |
| 4 | 39 | 0.320640 |
| 5 | 78 | 0.280810 |

All runs stopped before the epoch ceiling. The predeclared median-epoch rule gives **78 epochs** for subsequent full-training refitting. No final refit or test evaluation has been performed. The recorded winner is the best candidate evaluated; small differences between candidates do not establish statistical superiority.


## 8. Export reproducibility outputs
The archive contains analysis outputs and the search configuration, including fold checkpoints. The academic report is maintained separately and is excluded. In Colab this cell offers the archive for download.

In [ ]:
import zipfile

archive = Path("challenge_progress.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for directory in [OUTPUT_ROOT, Path("configs")]:
        for path in sorted(directory.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=str(path))
print(f"Saved {archive.resolve()}")
try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))

## Subsequent work
Review the selected configuration and convergence, then implement the sample-dependent exponent network and its physics-loss comparison. Final refitting, test metrics, comparison figures, and the separate academic report remain outstanding.